# Package

In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import os
import csv

# 检查是否有可用的 GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"当前使用的设备是: {device}")

# 固定随机种子
def same_seed(seed): 
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

same_seed(42) # 设一个你喜欢的数字
# !nvidia-smi

当前使用的设备是: cuda


# DataSet

## 数据构成
- 每行一个样本
- 第一列为 样本id
- 接下来34列是美国的州/地区缩写，例如 AL, AZ, CA, …, WI。每行只有一个州列为1，即 one-hot。
- 然后是18个调查变量，它们完整重复了3次，对应三次不同的调查时间点。
    - cli	是否有COVID‑like illness（类新冠症状）
    - ili	是否有Influenza‑like illness（类流感症状）
    - wnohh_cmnty_cli	社区中（不含同住者）出现类新冠症状的比例
    - wbelief_masking_effective	相信戴口罩有效的程度（百分比或0‑100分）
    - wbelief_distancing_effective	相信社交距离有效的程度
    - wcovid_vaccinated_friends	朋友中已接种疫苗的比例
    - wlarge_event_indoors	参加室内大型活动的频率
    - wothers_masked_public	在公共场合看到他人戴口罩的频率
    - wothers_distanced_public	在公共场合看到他人保持距离的频率
    - wshop_indoors	室内购物的频率
    - wrestaurant_indoors	在室内餐厅用餐的频率
    - wworried_catch_covid	担心自己感染COVID的程度
    - hh_cmnty_cli	同住家庭成员或社区中类新冠症状的比例
    - nohh_cmnty_cli	非家庭社区中类新冠症状的比例
    - wearing_mask_7d	过去7天戴口罩的频率
    - public_transit	使用公共交通的频率
    - worried_finances	担心财务状况的程度
    - tested_positive	目标变量：是否检测为阳性
- 值得注意的是，test set 相比于 train set 少了第三次的 tested_positive，即 我们要预测这次结果

In [2]:
class COVID19Dataset(Dataset):
    def __init__(self, path, mode='train', valid_ratio=0.2, seed=0):
        self.mode = mode

        self.feature_cols = [
            'tested_positive.1',
            'tested_positive',
            'hh_cmnty_cli',
            'nohh_cmnty_cli',
            'hh_cmnty_cli.1',
            'wnohh_cmnty_cli',
            'nohh_cmnty_cli.1',
            'wnohh_cmnty_cli.1',
            'hh_cmnty_cli.2',
            'nohh_cmnty_cli.2',
            'wnohh_cmnty_cli.2',
            'ili',
            'cli',
            'cli.1',
            'ili.1',
            'cli.2',
            'ili.2',
            'public_transit.2',
            'public_transit.1',
            'public_transit',
            'wlarge_event_indoors.2',
            'wlarge_event_indoors',
            'wlarge_event_indoors.1',
            'wrestaurant_indoors.2',
            'wrestaurant_indoors.1',
            'wrestaurant_indoors',
            'wshop_indoors.2',
            'wshop_indoors.1',
            'wshop_indoors'
        ]

        data = pd.read_csv(path)

        for col in self.feature_cols:
            if col not in data.columns:
                raise ValueError(f'Column "{col}" not found in {path}')

        if mode in ['train', 'dev']:
            target_col = data.columns[-1]
            x = data[self.feature_cols].values
            y = data[target_col].values

            n = len(data)
            torch.manual_seed(seed)
            indices = torch.randperm(n).tolist()

            train_size = int((1 - valid_ratio) * n)
            train_idx = indices[:train_size]
            dev_idx = indices[train_size:]

            if mode == 'train':
                self.x = torch.FloatTensor(x[train_idx])
                self.y = torch.FloatTensor(y[train_idx])
            else:
                self.x = torch.FloatTensor(x[dev_idx])
                self.y = torch.FloatTensor(y[dev_idx])

        elif mode == 'test':
            self.x = torch.FloatTensor(data[self.feature_cols].values)
            self.y = None

        else:
            raise ValueError("mode should be 'train', 'dev', or 'test'")

        self.dim = self.x.shape[1]

        print(f'Finished reading the {mode} set of COVID19 Dataset '
              f'({len(self.x)} samples found, each dim = {self.dim})')

    def __getitem__(self, idx):
        if self.mode in ['train', 'dev']:
            return self.x[idx], self.y[idx]
        return self.x[idx]

    def __len__(self):
        return len(self.x)

train_path = r"D:\MachineLearning\Hung-yi-Lee-ML-DL-HomeWork\HW1\ml2021spring-hw1\covid_train.csv"
test_path = r"D:\MachineLearning\Hung-yi-Lee-ML-DL-HomeWork\HW1\ml2021spring-hw1\covid_test.csv"

train_dataset = COVID19Dataset(train_path, mode='train')
val_dataset = COVID19Dataset(train_path, mode='dev')
test_dataset = COVID19Dataset(test_path, mode='test')

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

print("train x shape:", train_dataset.x.shape)
print("val x shape:", val_dataset.x.shape)
print("test x shape:", test_dataset.x.shape)

Finished reading the train set of COVID19 Dataset (2407 samples found, each dim = 29)
Finished reading the dev set of COVID19 Dataset (602 samples found, each dim = 29)
Finished reading the test set of COVID19 Dataset (997 samples found, each dim = 29)
train x shape: torch.Size([2407, 29])
val x shape: torch.Size([602, 29])
test x shape: torch.Size([997, 29])


# Model

In [7]:
class My_Model(nn.Module):
    def __init__(self, input_dim):
        super(My_Model, self).__init__()
        # 定义神经网络的层
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64), # 第一层：把输入维度映射到 64个神经元
            nn.ReLU(),                # 激活函数：加入非线性，让模型变聪明
            nn.Linear(64, 32),        # 第二层：64 映射到 32
            nn.ReLU(),
            nn.Linear(32, 1)          # 输出层：因为是预测一个具体数值(回归)，所以输出维度是 1
        )

    def forward(self, x):
        # 定义数据是怎么流过这些层的
        return self.net(x).squeeze(1) # squeeze 是为了把维度从 (batch_size, 1) 变成 (batch_size)

# Train

In [10]:
# 初始化模型、损失函数和优化器
input_dim = train_dataset.x.shape[1]
model = My_Model(input_dim).to(device) # 把模型放到 GPU 上

criterion = nn.MSELoss() # 均方误差 (Mean Squared Error) 作为损失函数
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=50, verbose=True
)

# 训练相关的参数设定
n_epochs = 3000          # 最大训练轮数
patience = 300           # 容忍度：如果验证集Loss连续 300 轮没降，就提前停止
best_loss = 100000.0     # 记录历史最低验证集Loss
early_stop_count = 0     # 记录没降的轮数

print("开始训练...")
for epoch in range(n_epochs):
    
    # === 训练阶段 ===
    model.train() # 设置为训练模式
    train_loss = []
    for x, y in train_loader:
        x, y = x.to(device), y.to(device) # 数据放入 GPU
        
        optimizer.zero_grad() # 1. 梯度清零
        pred = model(x)       # 2. 前向传播：模型预测
        loss = criterion(pred, y) # 3. 计算误差 (Loss)
        loss.backward()       # 4. 反向传播：计算梯度
        optimizer.step()      # 5. 更新参数
        
        train_loss.append(loss.item())
        
    # === 验证阶段 ===
    model.eval() # 设置为评估模式（关闭Dropout等）
    val_loss = []
    with torch.no_grad(): # 验证时不计算梯度，省内存并加速
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            pred = model(x)
            loss = criterion(pred, y)
            val_loss.append(loss.item())
            
    # 计算平均 loss
    avg_train_loss = sum(train_loss) / len(train_loss)
    avg_val_loss = sum(val_loss) / len(val_loss)
    scheduler.step(avg_val_loss)
    
    # 打印进度 (每 100 轮打印一次)
    if (epoch+1) % 100 == 0:
        print(f"Epoch: {epoch+1:4d} | Train Loss: {avg_train_loss:.5f} | Val Loss: {avg_val_loss:.5f}")
        
    # === 早停机制 (Early Stopping) ===
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), 'best_model.ckpt') # 保存表现最好的模型
        early_stop_count = 0
    else:
        early_stop_count += 1
        
    if early_stop_count >= patience:
        print(f"验证集 Loss 已经连续 {patience} 轮没下降，在第 {epoch+1} 轮提前停止！")
        break

开始训练...
Epoch:  100 | Train Loss: 1.14069 | Val Loss: 1.01547
Epoch:  200 | Train Loss: 1.04071 | Val Loss: 0.89038
Epoch:  300 | Train Loss: 0.96253 | Val Loss: 0.84914
Epoch:  400 | Train Loss: 0.91696 | Val Loss: 0.79350
Epoch:  500 | Train Loss: 0.90310 | Val Loss: 0.77729
Epoch:  600 | Train Loss: 0.92395 | Val Loss: 0.77986
Epoch:  700 | Train Loss: 0.99199 | Val Loss: 0.80966
Epoch:  800 | Train Loss: 0.91312 | Val Loss: 0.75181
Epoch:  900 | Train Loss: 0.87998 | Val Loss: 0.75133
Epoch: 1000 | Train Loss: 0.86999 | Val Loss: 0.75256
Epoch: 1100 | Train Loss: 0.86687 | Val Loss: 0.75110
Epoch: 1200 | Train Loss: 0.86161 | Val Loss: 0.74986
验证集 Loss 已经连续 300 轮没下降，在第 1204 轮提前停止！


# Inference

In [11]:
# 读取表现最好的模型
model = My_Model(input_dim).to(device)
model.load_state_dict(torch.load('best_model.ckpt'))

model.eval() # 设置为评估模式
preds = []

print("开始预测测试集...")
with torch.no_grad():
    for x in test_loader:
        x = x.to(device)
        pred = model(x)
        preds.append(pred.detach().cpu()) # 把预测结果移回 CPU

# 将所有批次的预测结果拼接起来
preds = torch.cat(preds, dim=0).numpy()

# === 生成 Kaggle 提交文件 ===
# 生成格式通常包含两列：id 和 predicted_value
with open('submission.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['id', 'tested_positive']) # 写入表头（根据Kaggle要求填写）
    for i, p in enumerate(preds):
        writer.writerow([i, p])

print("预测完毕！结果已保存到 submission.csv")

开始预测测试集...
预测完毕！结果已保存到 submission.csv


D:\Temp\ipykernel_35384\1023138332.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_model.ckpt'))
